# Import the necessary libraries

In [ ]:
# Adds the parent directory to sys.path
import sys
sys.path.append('../')

from problems.benchmark_problems import get_pareto
from problems.microgrid_function import simulation

from bisect import bisect
from pathlib import Path
from pygmo import fast_non_dominated_sorting # type: ignore

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import plotly.colors as pc
import plotly.graph_objects as go

# Microgrid Plotting

## Collecting Solutions From Each Batteries

In [ ]:
############ CUSTOMIZABLE PARAMETERS ############
position_dim = 3
objective_dim = 3

# Lead_Acid(0) Li-ion(1) ZEBRA(2) NaS(3) NiCd(4) NiMH(5) RFV(6) ZnBr(7)
bat_name = ['Lead_Acid', 'Li-ion', 'ZEBRA', 'NaS', 'NiCd', 'NiMH', 'RFV', 'ZnBr']
select_bat = [0,1,2,3,4,5,6,7]

# Choosing the files
choose_global_best_attribution_type = [0,1] # [0, 1, 2 ,3]
choose_dm_pool_type = [0,1,2] # [0, 1, 2]
choose_de_mutation_type = [0,1,2,3,4] # [0, 1, 2, 3, 4]
#################################################

# Name of chosen files
file_names = [
        [
        f"MESH_G{i+1}S{j+1}D{k+1}_{bat_name[b]}_3_{position_dim}.pkl"
        for i in choose_global_best_attribution_type
        for j in choose_dm_pool_type
        for k in choose_de_mutation_type
      ] for b in select_bat
    ]
plot_config_name = [f"G{i+1}S{j+1}D{k+1}" for _ in select_bat for i in choose_global_best_attribution_type for j in choose_dm_pool_type for k in choose_de_mutation_type]
plot_bat_name = ['Lead Acid', 'Li-ion', 'ZEBRA', 'NaS', 'NiCd', 'NiMH', 'RFV', 'ZnBr']

num_batteries = len(file_names) # Number of dataset

##### Color setup #####
cmap = plt.get_cmap("tab10")
solid_colors = [mcolors.to_hex(cmap(i+1)) for i in range(num_batteries)]
colorscales = pc.named_colorscales() # List of color scales
#######################

##### Marker setup #####
marker_symbols = ['circle', 'diamond', 'square', 'cross', 'x', 'diamond-open', 'square-open', 'circle-open']
num_symbols = len(marker_symbols)
#######################

# Collecting data from files
battery_name_idxs = [0]
config_name_idxs = [0]
count = 0
battery_solutions = np.empty((0,position_dim))
battery_pareto = np.empty((0,objective_dim))
for i in range(num_batteries):
	for j in range(len(file_names[i])):
		# Merging all solutions and pareto fronts for the same battery type
		with open("../results/" + file_names[i][j], "rb") as f:
			r = pickle.load(f)
			battery_solutions = np.vstack((battery_solutions, r['combined'][0]))
			battery_pareto = np.vstack((battery_pareto, r['combined'][1]))
			count += len(r['combined'][0])
			config_name_idxs.append(count)
	battery_name_idxs.append(count)
# Getting the non-dominated front indices
nds_idxs = fast_non_dominated_sorting(battery_pareto)[0][0]

## Pareto Front

In [ ]:
pareto_front = []

# Labeling the solutions with the battery names
for i in range(num_batteries):
	nds_battery_idxs = nds_idxs[np.where((nds_idxs >= battery_name_idxs[i]) & (nds_idxs < battery_name_idxs[i+1]))[0]]
	nds_battery_pareto = battery_pareto[nds_battery_idxs]
	print(f"Battery: {bat_name[select_bat[i]]}, Pareto solutions: {len(nds_battery_pareto)}")
	# Creating the plots
	pareto_front.append(
	go.Scatter3d(x=nds_battery_pareto[:,0],
				y=-nds_battery_pareto[:,1],
				z=nds_battery_pareto[:,2],
				mode='markers', marker=dict(size=4, color=solid_colors[i], symbol=marker_symbols[i]),
				name=plot_bat_name[select_bat[i]])
	)

fig = go.Figure()

for i in range(num_batteries):
    fig.add_trace(pareto_front[i])

axis_range = None

# Setting layout
fig.update_layout(
    shapes=[
        dict(
            type="rect",
            xref="paper", yref="paper",
            x0=0, y0=0, x1=1, y1=1,
            line=dict(color="black", width=2),
            fillcolor="rgba(0,0,0,0)"
        )
    ],
    scene=dict(
        camera=dict(
          eye=dict(x=1.5,y=1.5,z=1.5)
        ),
        xaxis=dict(
            title=dict(text='LCOE [US$/kWh]', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # X-axis background
            gridcolor='lightgray',     # Grid color on X axis 
            showbackground=True,       # Show background
            range=axis_range
        ),
        yaxis=dict(
            title=dict(text='RF', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # Y-axis background
            gridcolor='lightgray',     # Grid color on Y axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        zaxis=dict(
            title=dict(text='MEEF', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # Z-axis background
            gridcolor='lightgray',     # Grid color on Z axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        aspectmode='cube'
    ),
    legend=dict(
        x=0.95,
        y=0.95,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=14),
        itemsizing='constant'
    ),
    showlegend=True,
    # title=dict(
    #     text='MESH - ' + experiment_name,
    #     x=0.5,  # Centering the title
    #     y=0.95,
    #     xanchor='center',
    #     yanchor='top'
    # ),
    margin=dict(l=0, r=0, b=0, t=0), # Rearrange the margin
    width=800,   # Figure width (pixels)
    height=600,  # Figure height (pixels)
    paper_bgcolor="white",
    plot_bgcolor='white'
)

fig.show()

In [ ]:
nds_battery_pareto = battery_pareto[nds_idxs]
nds_battery_solutions = battery_solutions[nds_idxs]

color_values = np.linalg.norm(nds_battery_solutions, axis=1)

fig = go.Figure()

# Creating the plots
fig.add_trace(
	go.Scatter3d(x=nds_battery_pareto[:,0],
				 y=-nds_battery_pareto[:,1],
				 z=nds_battery_pareto[:,2],
				 mode='markers', marker=dict(size=4, color=color_values, colorscale=colorscales[21]),
				showlegend=False
        )
      )

axis_range = None

# Setting layout
fig.update_layout(
    shapes=[
        dict(
            type="rect",
            xref="paper", yref="paper",
            x0=0, y0=0, x1=1, y1=1,
            line=dict(color="black", width=2),
            fillcolor="rgba(0,0,0,0)"
        )
    ],
    scene=dict(
        camera=dict(
          eye=dict(x=1.5,y=1.5,z=1.5)
        ),
        xaxis=dict(
            title=dict(text='LCOE [US$/kWh]', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # X-axis background
            gridcolor='lightgray',     # Grid color on X axis 
            showbackground=True,       # Show background
            range=axis_range
        ),
        yaxis=dict(
            title=dict(text='RF', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # Y-axis background
            gridcolor='lightgray',     # Grid color on Y axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        zaxis=dict(
            title=dict(text='MEEF', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # Z-axis background
            gridcolor='lightgray',     # Grid color on Z axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        aspectmode='cube'
    ),
    legend=dict(
        x=0.95,
        y=0.95,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=14),
        itemsizing='constant'
    ),
    showlegend=True,
    # title=dict(
    #     text='MESH - ' + experiment_name,
    #     x=0.5,  # Centering the title
    #     y=0.95,
    #     xanchor='center',
    #     yanchor='top'
    # ),
    margin=dict(l=0, r=0, b=0, t=0), # Rearrange the margin
    width=800,   # Figure width (pixels)
    height=600,  # Figure height (pixels)
    paper_bgcolor="white",
    plot_bgcolor='white'
)

fig.show()

## Design Space

In [ ]:
design_space = []

# Labeling the solutions with the battery names
for i in range(num_batteries):
	nds_battery_idxs = nds_idxs[np.where((nds_idxs >= battery_name_idxs[i]) & (nds_idxs < battery_name_idxs[i+1]))[0]]
	nds_battery_solutions = battery_solutions[nds_battery_idxs]
	print(f"Battery: {bat_name[select_bat[i]]}, Design space solutions: {len(nds_battery_solutions)}")
	# Creating the plots
	design_space.append(
	go.Scatter3d(x=nds_battery_solutions[:,0],
				y=nds_battery_solutions[:,1],
				z=nds_battery_solutions[:,2],
				mode='markers', marker=dict(size=4, color=solid_colors[i], symbol=marker_symbols[i]),
				name=plot_bat_name[select_bat[i]])
	)

fig = go.Figure()

for i in range(num_batteries):
    fig.add_trace(design_space[i])

axis_range = None

# Setting layout
fig.update_layout(
    shapes=[
        dict(
            type="rect",
            xref="paper", yref="paper",
            x0=0, y0=0, x1=1, y1=1,
            line=dict(color="black", width=2),
            fillcolor="rgba(0,0,0,0)"
        )
    ],
    scene=dict(
        camera=dict(
          eye=dict(x=2,y=2,z=2)
        ),
        xaxis=dict(
            title=dict(text='Potência nominal do P. F. [kWp]', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # X-axis background
            gridcolor='lightgray',     # Grid color on X axis 
            showbackground=True,       # Show background
            range=axis_range
        ),
        yaxis=dict(
            title=dict(text='Potência nominal das turbinas [kW]', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # Y-axis background
            gridcolor='lightgray',     # Grid color on Y axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        zaxis=dict(
            title=dict(text='Capacidade da bateria [kWh]', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # Z-axis background
            gridcolor='lightgray',     # Grid color on Z axis
            showbackground=True,       # Show background
            range=axis_range
        )
    ),
    legend=dict(
        x=0.95,
        y=0.95,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=14),
        itemsizing='constant'
    ),
    showlegend=True,
    # title=dict(
    #     text='MESH - ' + experiment_name,
    #     x=0.5,  # Centering the title
    #     y=0.95,
    #     xanchor='center',
    #     yanchor='top'
    # ),
    margin=dict(l=0, r=0, b=0, t=0), # Rearrange the margin
    width=800,   # Figure width (pixels)
    height=600,  # Figure height (pixels)
    paper_bgcolor="white",
    plot_bgcolor='white'
)

fig.show()

In [ ]:
nds_battery_solutions = battery_solutions[nds_idxs]

color_values = np.linalg.norm(nds_battery_solutions, axis=1)

fig = go.Figure()

# Creating the plots
fig.add_trace(
	go.Scatter3d(x=nds_battery_solutions[:,0],
				 y=nds_battery_solutions[:,1],
				 z=nds_battery_solutions[:,2],
				 mode='markers', marker=dict(size=4, color=color_values, colorscale=colorscales[21]),
				showlegend=False
        )
      )

axis_range = None

# Setting layout
fig.update_layout(
    shapes=[
        dict(
            type="rect",
            xref="paper", yref="paper",
            x0=0, y0=0, x1=1, y1=1,
            line=dict(color="black", width=2),
            fillcolor="rgba(0,0,0,0)"
        )
    ],
    scene=dict(
        camera=dict(
          eye=dict(x=2,y=2,z=2)
        ),
        xaxis=dict(
            title=dict(text='Potência nominal do P. F. [kWp]', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # X-axis background
            gridcolor='lightgray',     # Grid color on X axis 
            showbackground=True,       # Show background
            range=axis_range
        ),
        yaxis=dict(
            title=dict(text='Potência nominal das turbinas [kW]', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # Y-axis background
            gridcolor='lightgray',     # Grid color on Y axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        zaxis=dict(
            title=dict(text='Capacidade da bateria [kWh]', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # Z-axis background
            gridcolor='lightgray',     # Grid color on Z axis
            showbackground=True,       # Show background
            range=axis_range
        )
    ),
    legend=dict(
        x=0.95,
        y=0.95,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=14),
        itemsizing='constant'
    ),
    showlegend=True,
    # title=dict(
    #     text='MESH - ' + experiment_name,
    #     x=0.5,  # Centering the title
    #     y=0.95,
    #     xanchor='center',
    #     yanchor='top'
    # ),
    margin=dict(l=0, r=0, b=0, t=0), # Rearrange the margin
    width=800,   # Figure width (pixels)
    height=600,  # Figure height (pixels)
    paper_bgcolor="white",
    plot_bgcolor='white'
)

fig.show()

## Filtering Solutions

In [ ]:
# Getting the individual optimized solutions in the non-dominated front
nds_battery_pareto = battery_pareto[nds_idxs]
nds_battery_solutions = battery_solutions[nds_idxs]
nds_min_idxs = np.argmin(nds_battery_pareto, axis=0)
indiv_solutions = nds_battery_pareto[nds_min_idxs]
global_idxs = nds_idxs[nds_min_idxs]

# Loading seasonal data
load = np.genfromtxt('../seasonal_data/load.txt')
temperature = np.genfromtxt('../seasonal_data/temperature.txt')
solar_data = np.genfromtxt('../seasonal_data/irradiance.txt')
wind_data = np.genfromtxt('../seasonal_data/wind.txt')

# Defining simulation function
microgrid_simulation = lambda args, select_bat: simulation(args[0], args[1], args[2], select_bat, load, temperature, solar_data, wind_data)

# Getting solution informations
objs = ['LCOE', 'RF', 'MEEF']
for obj in range(len(nds_min_idxs)):
	sol_idx = global_idxs[obj]
	# Finding which battery the solution belongs to
	bat_idx = select_bat[bisect(battery_name_idxs, sol_idx) - 1]
	# Finding which configuration the solution belongs to
	config_name_idx = bisect(config_name_idxs, sol_idx) - 1
	# Printing solution information
	print(f"Objective {obj+1} - {objs[obj]}:")
	print(f"  Battery Type: {plot_bat_name[bat_idx]}")
	print(f"  Configuration: {plot_config_name[config_name_idx]}")
	print(f"  Design Variables: {battery_solutions[sol_idx]}")
	print(f"  Objectives: {battery_pareto[sol_idx]}\n")
	# Running simulation for the optimal solution
	microgrid = microgrid_simulation(battery_solutions[sol_idx], bat_idx)
	microgrid.run()
	Path("./simulation_data").mkdir(parents=False, exist_ok=True)
	microgrid.logging(file_name=f"./simulation_data/{bat_name[bat_idx]}_{plot_config_name[config_name_idx]}_{objs[obj]}")


## Microgrid Simulation

### Plotting Function Definitions

In [ ]:
def plotBarChart(df, net_metering=True):
	data = [[], [], [], [], [], []]

	if net_metering:
		idx_net_surplus = 11
		label_net_surplus = 'Compensação'
		# label_net_surplus = 'Net Metering'
	else:
		idx_net_surplus = 12
		label_net_surplus = 'Desperdício'
		# label_net_surplus = 'Surplus'

	plt.figure(figsize=(8,6))
	plt.ylabel('Energia [MWh]', fontsize=16)
	# plt.ylabel('Energy [MWh]', fontsize=22)

	labels = ['Painéis (Sup.)', 'Turbinas (Sup.)', 'Bateria (Sup.)', 'Compra', label_net_surplus, 'Demanda']
	# labels = ['PV (Sup.)', 'Wind (Sup.)', 'ESS (Sup.)', 'Buy', label_net_surplus, 'Load'] 

	months = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
	# months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
	months_indexes = np.arange(len(months))
	bar_width = 0.4

	for i in range(0, 12):
		d = df.iloc[range(720*i, 720*(i+1)), [3, 4, 8, 9, idx_net_surplus, 0]].agg('sum')
		data[0] += [d.iloc[0]/1000] # PV supply
		data[1] += [d.iloc[1]/1000] # Wind supply
		data[2] += [d.iloc[2]/1000] # Ess discharge supply
		data[3] += [d.iloc[3]/1000] # Public Grid buy
		data[4] += [d.iloc[4]/1000] # Net Metering (compensated) or Surplus
		data[5] += [d.iloc[5]/1000] # Load

	b0 = plt.bar(months_indexes - bar_width / 2, data[0], width=bar_width, label=labels[0], color="yellow", hatch='o')
	b1 = plt.bar(months_indexes - bar_width / 2, data[1], width=bar_width, label=labels[1], color='green', hatch='*', bottom=data[0])
	b2 = plt.bar(months_indexes - bar_width / 2, data[2], width=bar_width, label=labels[2], color='blue', hatch='.',
				 bottom=[data[0][i]+data[1][i] for i in range(len(data[0]))])
	b3 = plt.bar(months_indexes - bar_width / 2, data[3], width=bar_width, label=labels[3], color='red', hatch='x',
				 bottom=[data[0][i]+data[1][i]+data[2][i] for i in range(len(data[0]))])
	b4 = plt.bar(months_indexes - bar_width / 2, data[4], width=bar_width, label=labels[4], color='orange', hatch='//',
				 bottom=[data[0][i]+data[1][i]+data[2][i]+data[3][i] for i in range(len(data[0]))])
	b5 = plt.bar(months_indexes + bar_width / 2, data[5], width=bar_width, label='Demanda', color='purple')

	plt.xticks(ticks=months_indexes, labels=months, fontsize=16)
	plt.yticks(fontsize=16)

	bar_list = [b0, b1, b2, b3, b4, b5]
	plt.legend(bar_list, labels, loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3, fontsize=14)
	plt.show()

def sankeyChart(df):
	labels = [
    "Geradores (Ger.)",			  # 0
    "Bateria (Sup.)",             # 1
    "Compra",                     # 2
    "Compensação",                # 3
    "Demanda"                     # 4
	]
	# labels = [
    # "Generators (Gen.)",	 		# 0
    # "Battery (Sup.)",             # 1
    # "Purchase",                   # 2
    # "Compensation",               # 3
    # "Load"                        # 4
	# ]

	pv_to_load = df['Photovoltaic Panel Supply [kWh]'].sum()
	wt_to_load = df['Wind Turbine Supply [kWh]'].sum()

	gen_to_batt = df['Battery Charge [kWh]'].sum()
	batt_to_load = df['Battery Supply [kWh]'].sum()

	gen_to_credit = df['Energy Credited [kWh]'].sum()
	credit_to_load = df['Energy Compensated [kWh]'].sum()
	grid_to_load = df['Energy Purchased [kWh]'].sum()

	source = [
		0, # Generation -> Load
		0, # Generation -> Battery (Charge)
		0, # Generation -> Credit
		1, # Battery (Discharge) -> Load
		2, # Grid -> Load
		3, # Credit -> Load
	]
	target = [
		4,  # Load
		1,  # Battery (Charge)
		3,  # Credit
		4,  # Load
		4,  # Load
		4   # Load
	]

	value = [pv_to_load + wt_to_load, gen_to_batt, gen_to_credit, batt_to_load, grid_to_load, credit_to_load]

	fig = go.Figure(
		go.Sankey(
			node=dict(label=labels, pad=15, thickness=20, color=['#9E9D24', 'blue', 'red', 'orange', 'purple']),
			link=dict(source=source,
					target=target,
					value=value,
					hovertemplate="%{value:.2f} kWh<extra></extra>"
			)
		)
	)
	fig.update_layout(font_size=16, width=800, height=600)
	fig.show()

def plotOperation(df, start_hour=0, hours=None, granularity="day"):
    """
    granularity: "hour", "day" ou "month"
    """

    # ----------------------------
    # Temporal slice
    # ----------------------------
    if hours is None:
        next_hours = len(df) - start_hour
    else:
        next_hours = hours
    pv = df.iloc[start_hour:start_hour + next_hours].copy()
    pv["hour_index"] = np.arange(len(pv))
    # ----------------------------
    # Plot
    # ----------------------------
    fig, ax = plt.subplots(figsize=(12, 8))

    # Energy sources
    ax.bar(
        pv["hour_index"], pv["Photovoltaic Panel Generation [kWh]"],
        width=1, align="edge", label="Painéis (Ger.)", alpha=0.7,
        color="yellow", hatch="o"
    )
    ax.bar(
        pv["hour_index"], pv["Wind Turbine Generation [kWh]"],
        width=1, align="edge", label="Turbinas (Ger.)", alpha=0.7,
        color="green", hatch="*"
    )
    ax.bar(
        pv["hour_index"], pv["Energy Purchased [kWh]"],
        width=1, align="edge", label="Compra", alpha=0.7,
        color="red", hatch="x"
    )
    ax.bar(
        pv["hour_index"], pv["Energy Compensated [kWh]"],
        width=1, align="edge", label="Compensação", alpha=0.7,
        color="orange", hatch="//"
    )

    # ----------------------------
    # Lines
    # ----------------------------
    ax.plot(
        pv["hour_index"], pv["Load [kWh]"],
        color="purple", lw=2, label="Demanda"
    )

    ax.plot(
        pv["hour_index"], pv["Battery State of Charge [kWh]"],
        color="blue", lw=2, linestyle="-.",
        label="Carga da Bateria"
    )

    # ----------------------------
    # X ticks (granularity)
    # ----------------------------
    if granularity == "hour":
        xticks = np.arange(0, len(pv), 1)
        xtick_labels = xticks
        xlabel = "Hora"

    elif granularity == "day":
        xticks = np.arange(0, len(pv), 24)
        xtick_labels = np.arange(1, len(xticks) + 1)
        xlabel = "Dia"

    elif granularity == "month":
        months = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun",
                   "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
        month_starts = [0, 24*31, 24*(31+28), 24*(31+28+31), 24*(31+28+31+30),
                        24*(31+28+31+30+31), 24*(31+28+31+30+31+30), 24*(31+28+31+30+31+30+31),
                        24*(31+28+31+30+31+30+31+31), 24*(31+28+31+30+31+30+31+31+30),
                        24*(31+28+31+30+31+30+31+31+30+31), 24*(31+28+31+30+31+30+31+31+30+31+30)]
        for i in range(len(month_starts)):
            if month_starts[i] > len(pv):
                month_starts = month_starts[:i]
                break
        xticks = np.array(month_starts)
        xtick_labels = months[:len(month_starts)]
        xlabel = "Mês"

    else:
        raise ValueError("Granularity must be 'hour', 'day' ou 'month'")

    ax.set_xticks(xticks)
    ax.set_xticklabels(xtick_labels)

    # ----------------------------
    # Aesthetics
    # ----------------------------
    ax.set_xlim(0, len(pv))
    ax.set_ylabel("Energia [kWh]", fontsize=16)
    ax.set_xlabel(xlabel, fontsize=16)

    ax.grid(axis="y", alpha=0.4)

    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.08),
        ncol=3,
        fancybox=True,
        shadow=True,
        fontsize=16
    )

    ax.tick_params(axis="both", labelsize=14)
    fig.tight_layout()
    plt.show()

### Plotting

In [ ]:
############ CUSTOMIZABLE PARAMETERS ############
select_bat = 0 # Lead_Acid(0) Li-ion(1) ZEBRA(2) NaS(3) NiCd(4) NiMH(5) RFV(6) ZnBr(7)
use_metering = True # Using (True), not using (False)
objective = 2 # LCOE(0), RF(1), MEEF(2)
G = 1
S = 1
D = 2
#################################################

objs = ['LCOE', 'RF', 'MEEF']

# File name
filename = f'{bat_name[select_bat]}_G{G}S{S}D{D}_{objs[objective]}.xlsx'
simulation_data = pd.read_excel('./simulation_data/' + filename, sheet_name="Microgrid Power Flow", usecols="A:M").abs()

#plotBarChart(simulation_data, net_metering=use_metering)
#sankeyChart(simulation_data)
plotOperation(simulation_data, start_hour=0, hours=None, granularity="month")

# Benchmark 2D:

In [ ]:
objective_dim = 2 # Number of objectives
position_dim = 20 # Number of variables
n_pareto_points = 5000 # Number of Pareto points to be generated
wfg_k = 6 # The parameter k for WFG problems

experiment_name = "zdt4"

# Choosing the files
insert_mopso_cd = True
insert_nsga2 = True
insert_nspso = False
insert_spea2 = True


file_tuple = []

# MESH files
file_tuple.append((f"MESH_{experiment_name}_{objective_dim}_{position_dim}.pkl", 'MESH'))
 
if insert_mopso_cd:
    file_tuple.append((f'MOPSO_CD_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'MOPSO-CD'))
if insert_nsga2:
  	file_tuple.append((f'NSGA2_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'NSGA-II'))
if insert_nspso:
    file_tuple.append((f'NSPSO_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'NSPSO'))
if insert_spea2:
    file_tuple.append((f'SPEA2_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'SPEA-II'))

num_datasets = len(file_tuple) # Number of dataset

##### Color setup #####
cmap = plt.get_cmap("tab10")
solid_colors = [mcolors.to_hex(cmap(i+1)) for i in range(num_datasets)]
colorscales = pc.named_colorscales() # List of color scales
#######################

##### Marker setup #####
marker_symbols = ['circle', 'diamond', 'square', 'cross', 'x', 'circle-open', 'diamond-open', 'square-open']
num_symbols = len(marker_symbols)
#######################

pareto_front = []
for i in range(num_datasets):
  with open("../results/" + file_tuple[i][0], "rb") as f:
    r = pickle.load(f)
    pareto_front.append(
      go.Scatter(x=r['combined'][1][:,0],
                   y=r['combined'][1][:,1],
                   mode='markers', 
                   marker=dict(size=8, symbol=marker_symbols[i % num_symbols], color=solid_colors[i]), #r['combined'][1][:,1], colorscale=colorscales[(i+1) % num_scales]),
                   name=file_tuple[i][1],
                   showlegend=True)
    )

## Plotting

In [ ]:
# Creating a subplot
fig = go.Figure()

for i in range(num_datasets):
    fig.add_trace(pareto_front[i])

axis_range = None # [-0.05, 1.05]
# Setting layout
fig.update_layout(
    shapes=[
        dict(
            type="rect",
            xref="paper", yref="paper",
            x0=0, y0=0, x1=1, y1=1,
            line=dict(color="black", width=2),
            fillcolor="rgba(0,0,0,0)"  # Transparente
        )
    ],
    xaxis=dict(title=dict(text='f1', font=dict(size=14)), tickfont=dict(size=14)),
    yaxis=dict(title=dict(text='f2', font=dict(size=14)), tickfont=dict(size=14)),
    # title=dict(
    #     text='MESH - ' + experiment_name.upper(),
    #     x=0.5,  # Centering the title
    #     xanchor='center',
    #     yanchor='top'
    # ),
    legend=dict(
        x=0.99,
        y=0.99,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",  # fundo leve para não colidir com os pontos
        bordercolor="black",
        borderwidth=1,
        font=dict(size=14)
    ),
    showlegend=True,
    margin=dict(l=5, r=5, b=5, t=5), # Rearrange the margin
    width=600,   # Figure width (pixels)
    height=480,  # Figure height (pixels)
    paper_bgcolor="white",
    plot_bgcolor='white'
)

# Plotting the Pareto front
real_pareto_front = get_pareto(experiment_name, n_pareto_points, position_dim, 2, wfg_k)
fig.add_trace(
	go.Scatter(x=real_pareto_front[:,0],
		y=real_pareto_front[:,1],
		mode='markers', marker=dict(size=2, color='black'),
		name='Pareto Front',
		showlegend=True
		)
	)

# Showing the figure
fig.show()

# Benchmark 3D:

In [ ]:
objective_dim = 3 # Number of objectives
position_dim = 20 # Number of variables
n_pareto_density = 100 # Number of Pareto points to be generated
wfg_k = 6 # The parameter k for WFG problems

experiment_name = "dtlz4"

# Choosing the files
insert_mopso_cd = True
insert_nsga2 = True
insert_nspso = False
insert_spea2 = True


file_tuple = []

# MESH files
file_tuple.append((f"MESH_{experiment_name}_{objective_dim}_{position_dim}.pkl", 'MESH'))
 
if insert_mopso_cd:
    file_tuple.append((f'MOPSO_CD_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'MOPSO-CD'))
if insert_nsga2:
  	file_tuple.append((f'NSGA2_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'NSGA-II'))
if insert_nspso:
    file_tuple.append((f'NSPSO_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'NSPSO'))
if insert_spea2:
    file_tuple.append((f'SPEA2_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'SPEA-II'))

num_datasets = len(file_tuple) # Number of dataset

##### Color setup #####
cmap = plt.get_cmap("tab10")
solid_colors = [mcolors.to_hex(cmap(i+1)) for i in range(num_datasets)]
colorscales = pc.named_colorscales() # List of color scales
#######################

##### Marker setup #####
marker_symbols = ['circle', 'diamond', 'square', 'cross', 'x', 'circle-open', 'diamond-open', 'square-open']
num_symbols = len(marker_symbols)
#######################

pareto_front = []
for i in range(num_datasets):
  with open("../results/" + file_tuple[i][0], "rb") as f:
    r = pickle.load(f)
    pareto_front.append(
      go.Scatter3d(x=r['combined'][1][:,0],
                   y=r['combined'][1][:,1],
                   z=r['combined'][1][:,2],
                   mode='markers', marker=dict(size=5, symbol=marker_symbols[i % num_symbols], color=solid_colors[i]), #r['combined'][1][:,1], colorscale=colorscales[(i+1) % num_scales]),
                   name=file_tuple[i][1],
                   showlegend=True)
    )

## Plotting

In [ ]:
# Creating a subplot
fig = go.Figure()

for i in range(num_datasets):
    fig.add_trace(pareto_front[i])

axis_range = None # [-0.1, 1.2]

# Setting layout
fig.update_layout(
    shapes=[
        dict(
            type="rect",
            xref="paper", yref="paper",
            x0=0, y0=0, x1=1, y1=1,
            line=dict(color="black", width=2),
            fillcolor="rgba(0,0,0,0)"  # Transparente
        )
    ],
    scene=dict(
        camera=dict(
          eye=dict(x=1,y=1,z=1)
        ),
        xaxis=dict(
            title=dict(text='f1', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',  # X-axis background
            gridcolor='lightgray',    # Grid color on X axis 
            showbackground=True,       # Show background
            range=axis_range
        ),
        yaxis=dict(
            title=dict(text='f2', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',  # Y-axis background
            gridcolor='lightgray',    # Grid color on Y axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        zaxis=dict(
            title=dict(text='f3', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',  # Z-axis background
            gridcolor='lightgray',    # Grid color on Z axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        aspectmode='cube'
    ),
    legend=dict(
        x=0.99,
        y=0.99,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",  # fundo leve para não colidir com os pontos
        bordercolor="black",
        borderwidth=1,
        font=dict(size=14)
    ),
    showlegend=True,
    # title=dict(
    #     text='MESH - ' + experiment_name,
    #     x=0.5,  # Centering the title
    #     xanchor='center',
    #     yanchor='top'
    # ),
    margin=dict(l=5, r=5, b=5, t=5), # Rearrange the margin
    width=600,   # Figure width (pixels)
    height=420,  # Figure height (pixels)
    paper_bgcolor="white",
    plot_bgcolor='white'
)

# Plotting the Pareto front
real_pareto_front = get_pareto(experiment_name, n_pareto_density, position_dim, 3, wfg_k)
fig.add_trace(
	go.Scatter3d(x=real_pareto_front[:,0],
		y=real_pareto_front[:,1],
		z=real_pareto_front[:,2],
		mode='markers', marker=dict(size=2, color='black'),
		name='Pareto Front',
		showlegend=True
		)
	)

# Showing the figure
fig.show()